# Julia notebook to *use* the symbolic solution to the frictional geostrophic equations with a specified relaxation buoyancy field, surface buoyancy field, and surface wind stress.

twnh Jun '26

This notebook uses the Green's function solution to:

\begin{align}
- f  v & = - \frac{\partial p}{\partial x} + Ek \frac{d^2 u}{d z^2}, \\
  f  u & = - \frac{\partial p}{\partial y} + Ek \frac{d^2 v}{d z^2} , 
\end{align}
for $u(z), v(z)$, given boundary conditions
\begin{align}
\text{Surface~}z = 0:
\begin{cases}
\displaystyle \partial_z u & = \tau^x \\
\displaystyle \partial_z v & = \tau^y 
\end{cases}
 \\
\text{Bottom~}z = -H:
\begin{cases}
u & = 0  \\
v & = 0 
\end{cases} ,
\end{align}
where $(\tau^x, \tau^y)$ is the surface wind stress and $Ek$ is the Ekman number.
The pressure $p$ is due to the surface pressure field $p_s(x,y)$ and buoyancy field $b(x,y,z)$:

\begin{align}
p(x,y,z) & = p_s(x,y) + \int_{z}^{0} b(x,y,z') \; dz' , \\
\implies p_b(x, y) & \equiv p(x, y, z=-H) = p_s(x,y) + \int_{-H(x,y)}^{0} b(x,y,z') \; dz' .
\end{align}

This code builds the equation satisfied by the surface pressure field $p_s(x,y)$ using the Green's function solution.
It then solves for the pressure field given a specified windstress and buoyancy forcing.

The buoyancy forcing comes from solving

\begin{align}
Ek \frac{d^2 b}{d z^2} + \Gamma \left( B(z) - b \right) = 0
\end{align}
for $b(z)$, given fixed diffusivity $\kappa$, and boundary conditions
\begin{align}
\text{Surface~}z = 0: b &= b_s ,\\
\text{Bottom~}z = -H: \frac{\partial b}{\partial z} &= 0
\end{align}
where $B(z)$ is the specified relaxation buoyancy field, $b_s$ is the specified surface buoyancy field, and $\Gamma$ is the relaxation rate. Again, this problem is solved using a Green's function that is specified in this notebook.

This notebook is based on `ExampleSolution_v_0.10.ipynb` with updated (slightly different) analytical results.

In [ ]:
notebook_name = "PG_Solution_v0.1"
using Symbolics
using ForwardDiff
@variables I        # Placeholder variable for imaginary unit, otherwise Symbolics.jl computes things like sqrt(im) as a numerical value.
using Gridap
using GridapGmsh
using WriteVTK

using Infiltrator

### Define symbols:

In [ ]:
# Geometry symbolic parameters:
@variables z::Real ξ::Real      # Both (z, ξ) in [-H, 0]
@variables x::Real y::Real
@variables H(x,y)::Real         # Function H(x, y), depth

# Frictional geostrophic equation symbolic parameters:
@variables Ek::Real             # Ekman number
@variables β::Real              # Coriolis parameter gradient β
@variables f(x,y)::Real         # Coriolis parameter f(y)
@variables ϕ(x,y)::Real
@variables τs(x,y)              # Complex surface wind stress τs(x, y)
@variables psg(x,y)             # Complex surface pressure gradient (∂/∂x + i ∂/∂y) p_s(x, y)
@variables b(x,y,z)::Real       # Buoyancy field b(x, y, z)

# Buoyancy equation symbolic parameters:
@variables ψ(x,y)::Real Γ::Real
@variables Bξ(x,y,ξ)::Real

# Differential operators:
∂x = Differential(x)
∂y = Differential(y)
∂z = Differential(z)

#### Function to build callable function from `Symbolics` expression. This is where `I` gets replaced with $\sqrt{-1}$:

In [ ]:
function replace_imag_unit(expr)
	I_vars = filter(v -> string(v) == "I", Symbolics.get_variables(expr))				# Check if imaginary unit variable is present and substitute
	if !isempty(I_vars)
    	tmp1 = substitute(expr, Dict(first(I_vars) => im))
	else
	    tmp1 = expr
	end
	return tmp1
end

function build_Symbolics_fn(expr,varnames)
	vars = [Symbolics.variable(Symbol(n)) for n in varnames]							# Extract Symbolics variables from variable names
	tmp1 = replace_imag_unit(expr)														# Replace imaginary unit variable with Julia's im
	tmp2 = expand_derivatives(tmp1)														# Expand derivatives
	tmp3vec = build_function(tmp2, vars; expression=Val{false})
	
	# Wrap function to throw an error if it's called with multiple scalar arguments rather than a vector:
	function tmp3(args::AbstractVector)
    	length(args) == length(vars) || throw(ArgumentError("Input must be vector of length $(length(vars))."))
    	tmp3vec(args)
	end
	tmp3(args...) = throw(MethodError(tmp3, args))
	return tmp3
end

### Define parameter values:

In [ ]:
# Define the parameter values
β_val = 0.4
f_val(x,y) = 1 + β_val * y
@info "Notebook accommodates a β-plane"
Γ_val = 0.2
Ek_val = 0.02

# Compute compound parameter:
ϕ_val(x,y) = sqrt(f_val(x,y) / Ek_val)
root_iϕ = I^(1//2) * ϕ
ψ_val = sqrt(Γ_val / Ek_val)
Ekman_depth = sqrt(Ek_val)

# Define domain depth, and hence the geometry and mesh:
# Double-basin Gaussian topography:
GmshMeshFileName = "reservoir.msh"
include("build_mesh.jl")
H_val(x,y) = H_Gaussian(x,y)
res = 0.1
build_geometry_and_mesh(H_val, res, GmshMeshFileName)

# Rectangular basin with flat bottom:
# GmshMeshFileName = "rectangle_v0.1.msh"
# R = 2.0 # Domain half-width
# H_val(x,y) = 1.0

# Define wind stress:
# τs_val(x,y) = 0.1 + 0.0I
# function τs_val(x, y)         # Circular wind stress pattern
#     r = sqrt(x^2 + y^2)
#     V = 0.0 * r^2 
#     return -V * y / r + I* V * x / r
# end
function τs_val(x, y)           # Sinusoidal zonal wind stress pattern
    return 0.1*sin(π*y/2R) + 0.0 * I
end

# Define relaxation and surface buoyancy fields:
B_val(x,y,ξ)  = ξ
bs_val(x,y)   = 0.0

# Define dictionaries for substitutions:
param_values = Dict(f=>f_val(x,y), ϕ=>ϕ_val(x,y), Γ=>Γ_val, ψ=>ψ_val, Ek => Ek_val)
fn_values = Dict(
    τs => τs_val(x, y),
    H  => H_val(x, y),
    Bξ => B_val(x, y, ξ),
    ∂x(H) => expand_derivatives(∂x(H_val(x, y))),
    ∂y(H) => expand_derivatives(∂y(H_val(x, y))),
)

# Report
println()
println("This case parameter values:")
display(param_values)

println()
println("This case function values:")
display(fn_values)

println()
println("Non-dimensional Ekman_depth (based on y=0) = $(Ekman_depth)")

### Compute the velocity G's function and various special cases of it:

In [ ]:
function define_Guv()                   # Taken from Latex notes on 30Jul26.
    prefactor = exp(-root_iϕ * (z + ξ)) / (2 * root_iϕ * (1 + exp(2 * root_iϕ * H)))
    Guv = ifelse(z <= ξ,
          exp(2 * root_iϕ * ξ) - exp(2 * root_iϕ * (z + H)) - exp(2 * root_iϕ * (z + ξ + H)) + 1,
          exp(2 * root_iϕ * z) - exp(2 * root_iϕ * (ξ + H)) - exp(2 * root_iϕ * (z + ξ + H)) + 1)
    Guv = prefactor * Guv
    return Guv
end

function define_Guv0()                  # Taken from Latex notes on 30Jul26.
    return ((exp(- root_iϕ * z))/(root_iϕ * (1 + exp(2 * root_iϕ * H)))) * (1 - exp(2 * root_iϕ * (z + H)))
end

function define_Guv_int_wrt_ξ()         # Taken from Latex notes on 30Jul26.
    return (exp(root_iϕ * (z + H)) - exp(root_iϕ * (H - z)) - exp(2 * root_iϕ * H) - 1) / (root_iϕ^2 * (1 + exp(2 * root_iϕ * H)))
end

Guv = define_Guv()
Guv0 = define_Guv0()
Guv_int_wrt_ξ = define_Guv_int_wrt_ξ()

### Compute the buoyancy G's function and various special cases of it:

In [ ]:
function define_Gb()                   # Taken from Latex notes on 30Jul26.
    prefactor = exp(ψ * (z + ξ)) / (2 * ψ * (1 + exp(2 * ψ * H)))
    Gb = ifelse(z <= ξ,
          (1 - exp(-2 * ψ * ξ)) * (exp(2 * ψ * H) + exp(-2 * ψ * z)),
          (1 - exp(-2 * ψ * z)) * (exp(2 * ψ * H) + exp(-2 * ψ * ξ)))
    Gb = prefactor * Gb
    return Gb
end

function define_∂ξ_Gb_ξ0()             # Taken from Latex notes on 30Jul26.
    return (exp(ψ * z) * (exp(2 * ψ * H) + exp(-2 * ψ * z))) / (1 + exp(2 * ψ * H))
end

Gb = define_Gb()
∂ξ_Gb_ξ0 = define_∂ξ_Gb_ξ0()

### Compute the buoyancy field function:

In [ ]:
function define_S_fn()           # Taken from Latex notes on 30Jul26.
    return (1/(root_iϕ * Ek))*tanh(root_iϕ * H)
end

function define_T_fn()           # Taken from Latex notes and also from ExampleSolution_v0.5.ipynb on 13Dec25.

    # Construct the buoyancy related T(x) term (excludes the stress-driven part):
    Latex_Txb_term = 0
    
    # Add the wind-stress driven part:
    T = Latex_Txb_term - (2 * exp(root_iϕ * H)/(1 + exp(2 * root_iϕ*H))) * τs
    return T
end

function define_A_fn()           # Taken from Latex notes on 30Jul26.
    A = (1 / (root_iϕ^2 * Ek)) * ((exp(2 * root_iϕ * H) - 1)/(root_iϕ * (1 + exp(2 * root_iϕ * H))) - H)
    return A
end

function define_B_fn()           # Taken from Latex notes and also from ExampleSolution_v0.5.ipynb on 13Dec25.
   
    # Construct the buoyancy related B(x) term (excludes the stress-driven part):
    Latex_Bxb_term = 0

    # Add the wind-stress driven part:
    B = Latex_Bxb_term + (exp(root_iϕ * H) - 1)^2 * τs
    # Final overall scaling:
    B = B / (root_iϕ^2 * (1 + exp(2 * root_iϕ * H)))
    return B
end

### Solve for surface pressure using Gridap

In [ ]:
println()
@time "Solving for the surface pressure using Gridap..." begin
# 1. Define the mesh
    model = GmshDiscreteModel(GmshMeshFileName)
    Ω = Gridap.Geometry.Triangulation(model)
    dΩ = Measure(Ω, 2)

# 2. Define the finite element space (piecewise linear, Dirichlet zero BC)
    order = 2
    reffe_scalar = ReferenceFE(lagrangian, Float64, order)
    V     = TestFESpace(model, reffe_scalar; conformity=:H1, dirichlet_tags="boundary")
    U     = TrialFESpace(V)
    reffe_vector = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
    V_vec = TestFESpace(model, reffe_vector; conformity=:H1, dirichlet_tags="boundary")

# 3. Define the coefficients of the elliptic equation:
    A_fn_tmp0 = substitute(substitute(define_A_fn(),param_values),fn_values)
    A_fn_tmp = build_Symbolics_fn(A_fn_tmp0, [x, y])
    A_fn(xx) = (typeof(xx[1]) <: Real ? A_fn_tmp([xx[1],xx[2]]) : 1.0)     # Might get called with non-Float argument

    B_fn_tmp0 = substitute(substitute(define_B_fn(),param_values),fn_values)
    B_fn_tmp = build_Symbolics_fn(B_fn_tmp0, [x, y])
    B_fn(xx) = (typeof(xx[1]) <: Real ? B_fn_tmp([xx[1],xx[2]]) : 0.0)     # Might get called with non-Float argument

# 4. Define weak form (variational formulation)
    a(u,v) = ∫( real( (∇(v) ⋅ VectorValue( 1.0, -1im)) * (A_fn * (∇(u) ⋅ VectorValue(1.0, 1im))) ) )dΩ
    l(v)   = ∫( real( (∇(v) ⋅ VectorValue(-1.0,  1im)) *  B_fn ) )dΩ

# 5. Assemble and solve
    op = AffineFEOperator(a, l, U, V)
    psurf = Gridap.solve(op)  # This is your numerical solution as a Gridap FEFunction
end

#### Solve for streamfunction of depth-integrated flow:

Fast assembly of complex-valued function using cached intermediate functions and function composition, not function calls inside the integral.
Need to interpolate real and imaginary parts separately (Gridap does not support complex-valued FE functions directly).

See: [this Gridap gitter thread](https://matrix.to/#/!mSZoaZwNZhWulNruaK:gitter.im/$EWrGYx6lcBOQt05k5y2EGXangkwEitmxwFjaslhlSt8?via=gitter.im&via=matrix.org&via=mozilla.org)

In [ ]:
println()
@time "Computing streamfunction for vertically-integrated horizontal velocity field using Gridap..." begin
    reffe_vector = ReferenceFE(lagrangian, VectorValue{2,Float64}, order)
    V_vec = TestFESpace(model, reffe_vector; conformity=:H1, dirichlet_tags="boundary")
    gradpsurf_projection = interpolate(∇(psurf), V_vec)

    ReA_fe = interpolate(x -> real(A_fn(x)), U)
    ImA_fe = interpolate(x -> imag(A_fn(x)), U)
    ReB_fe = interpolate(x -> real(B_fn(x)), U)
    ImB_fe = interpolate(x -> imag(B_fn(x)), U)
    AU_fe = (ReA_fe + 1im*ImA_fe)
    BU_fe = (ReB_fe + 1im*ImB_fe)
    dot_gradp = gradpsurf_projection ⋅ VectorValue(1, 1im)
    C_fn = AU_fe * dot_gradp + BU_fe

    a2(u,v) = ∫( ∇(v) ⋅ ∇(u))dΩ
    l2(v)   = ∫( - real( (∇(v) ⋅ VectorValue(1im, 1.0)) *  C_fn ) )dΩ
    op2 = AffineFEOperator(a2, l2, U, V)
    Psi = Gridap.solve(op2)
    writevtk(Ω,notebook_name * "_solution.vtu",cellfields=["psurf"=>psurf, "Psi"=>Psi, "ReA"=>ReA_fe, "ImagA"=>ImA_fe])
end

#### Setup helper functions:

In [ ]:
function ps_val(xx, yy)
    try
    	gp = evaluate(psurf, Gridap.Point(xx, yy))
	    return gp
    catch
        return 0.0
    end
end

function psg_val(xx, yy)
    try
	    tmp = evaluate(∇(psurf), Gridap.Point(xx, yy))
	    return tmp[1] + 1im*tmp[2] 
    catch
        return 0.0 + 1im*0.0
    end
end

function UV_val(xx,yy) 
    try
        psg_tmp = evaluate(∇(psurf), Gridap.Point(xx, yy))
        UV = A_fn_tmp([xx,yy])*(psg_tmp[1] + 1im*psg_tmp[2]) + B_fn_tmp([xx,yy])
        return UV
    catch
        return 0.0 + 1im*0.0
    end
end

function Psi_val(xx,yy)
    try
        Psi_tmp = evaluate(Psi, Gridap.Point(xx, yy))
        return Psi_tmp
    catch
        return 0.0
    end
end

### Solve for velocity field using the Green's function:

In [ ]:
# Use the portable B23 kernels generated from ComputeQuadratures.ipynb.
include(normpath(joinpath(pwd(), "..", "TheoryNotes", "VelocityFieldPrototype.jl")))
using .VelocityFieldPrototype

# The current nondimensional model uses uniform nu = kappa = 1.
velocity_parameters = VelocityParameters(Ek_val, Γ_val, 1.0, 1.0, 1.0, β_val)
velocity_kernels = load_symbolic_kernels()
H_x_val(xx, yy) = ForwardDiff.derivative(xq -> H_val(xq, yy), xx)
H_y_val(xx, yy) = ForwardDiff.derivative(yq -> H_val(xx, yq), yy)
τs_numeric(xx, yy) = ComplexF64(Symbolics.value(replace_imag_unit(τs_val(xx, yy))))
psg_b23(xx, yy) = psg_val(xx, yy)
velocity_gradient_step = 1e-5
@warn "The current surface-pressure RHS omits the B_val and bs_val forcing terms. The B23 velocity and buoyancy outputs include them, but a fully coupled run requires adding those terms to define_B_fn and define_T_fn."

S_fn_tmp		 = build_Symbolics_fn(substitute(substitute(define_S_fn(), param_values),fn_values), [x, y])
T_fn_tmp		 = build_Symbolics_fn(substitute(substitute(define_T_fn(), param_values),fn_values), [x, y])
τb_fn(xx,yy)     = S_fn_tmp([xx,yy])*psg_val(xx,yy) + T_fn_tmp([xx,yy])

pbarog_term = 0
pbarog_term_fn = build_Symbolics_fn(substitute(substitute(pbarog_term,param_values),fn_values), [x, y, z])

# Nx, Ny, Nz = 256, 257, 64
Nx, Ny, Nz = 128, 65, 32
xs     = range(-R,  R, Nx)
ys     = range(-R,  R, Ny)
zs     = range(-1,  0, Nz)
us     = fill(NaN, Nx, Ny, Nz)
vs     = fill(NaN, Nx, Ny, Nz)
bs     = fill(NaN, Nx, Ny, Nz)
Us     = fill(NaN, Nx, Ny)
Vs     = fill(NaN, Nx, Ny)
τxs    = fill(NaN, Nx, Ny)
τys    = fill(NaN, Nx, Ny)
τbxs   = fill(NaN, Nx, Ny)
τbys   = fill(NaN, Nx, Ny)
pss    = fill(NaN, Nx, Ny)
spds   = fill(NaN, Nx, Ny)
Psis   = fill(NaN, Nx, Ny)
Hs     = fill(NaN, Nx, Ny)
fs     = fill(NaN, Nx, Ny)
Hofs   = fill(NaN, Nx, Ny)

println()
@time "Computing velocity field from B23 Green-function kernels:" begin
for (ix, xx) in enumerate(xs)
	for (iy, yy) in enumerate(ys)
		this_H = H_val(xx, yy)
		# B23 uses centered horizontal differences for the remaining B terms.
		# Leave the one-stencil-wide edge undefined instead of evaluating outside the basin.
		stencil_is_wet = all(H_val(xq, yq) > 0.0 for (xq, yq) in (
			(xx - velocity_gradient_step, yy), (xx + velocity_gradient_step, yy),
			(xx, yy - velocity_gradient_step), (xx, yy + velocity_gradient_step),
		))
		if this_H > 0.0 && stencil_is_wet
			this_pss_val = ps_val( xx, yy)      # Interpolate or evaluate ps
			this_psg_val = psg_b23(xx, yy)
			this_τs_val  = τs_numeric(xx, yy)
			for (iz, zz) in enumerate(zs)
				if zz <= 0 && zz >= -this_H 
					this_uv_value = velocity_b23_from_gradps(
						velocity_kernels, velocity_parameters, H_val, psg_b23,
						τs_numeric, B_val, bs_val, H_x_val, H_y_val, xx, yy, zz;
						gradient_step=velocity_gradient_step,
					)
					us[ix, iy, iz] = float(real(this_uv_value))
					vs[ix, iy, iz] = float(imag(this_uv_value))
					bs[ix, iy, iz] = buoyancy_from_forcing(
						velocity_parameters, H_val, B_val, bs_val, xx, yy, zz,
					)
				end
			end
			# Compute 2D fields here:
			UV             = UV_val(xx,yy)
			Us[    ix, iy] = float(real(UV))
			Vs[    ix, iy] = float(imag(UV))
			spds[  ix, iy] = sqrt(Us[ix, iy]^2 + Vs[ix, iy]^2)
			τxs[   ix, iy] = float(real(this_τs_val))
			τys[   ix, iy] = float(imag(this_τs_val))
			τb_val         = τb_fn(xx,yy)
			τbxs[  ix, iy] = float(real(τb_val))
			τbys[  ix, iy] = float(imag(τb_val))
			pss[   ix, iy] = this_pss_val
			Psis[  ix, iy] = Psi_val(xx,yy)
			Hs[    ix, iy] = this_H
			fs[    ix, iy] = f_val(xx,yy)
			Hofs[  ix, iy] = this_H/f_val(xx,yy)
		end
	end
end
end

# Write out solution for display by Paraview:
vtk_grid(notebook_name * "_3D_solution.vti", xs, ys, zs) do vtk
	vtk["u_speed"]  = us
	vtk["v_speed"]  = vs
	vtk["b_field"]  = bs
end

vtk_grid(notebook_name * "_2D_solution.vti", xs, ys) do vtk
	vtk["U_speed"]      = Us
	vtk["V_speed"]      = Vs
	vtk["speed"]        = spds
	vtk["x_sfc_stress"] = τxs
	vtk["y_sfc_stress"] = τys
	vtk["x_bot_stress"] = τbxs
	vtk["y_bot_stress"] = τbys
	vtk["sfc_p"]        = pss
	vtk["Psi"]          = Psis
	vtk["Depth"]        = Hs
	vtk["Coriolis_f"]   = fs
	vtk["H_over_f"]     = Hofs
end